# Tokenization smoke test

Run this from the repo root in Colab or a local Jupyter environment. This notebook checks that the uncased tokenizer still emits normal BERT IDs while a parallel `capitalization_ids` feature preserves first-cap and all-caps information. It intentionally avoids model loading so it stays lightweight.


In [ ]:
from pathlib import Path
import os

COLAB_REPO = Path("/content/drive/MyDrive/Github/CapitalizationEmbeddings")
try:
    from google.colab import drive

    if not COLAB_REPO.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if COLAB_REPO.exists():
    os.chdir(COLAB_REPO)

print("repo:", Path.cwd())
%pip install -q -e . -r requirements-colab.txt


In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the CapitalizationEmbeddings repo root.")

In [ ]:
from transformers import AutoTokenizer

from capitalization_embeddings import tokenize_with_capitalization

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

examples = [
    "Tom met tom and TOM near iPhone HQ.",
    "NASA hired Alice in New York.",
]

batch = tokenize_with_capitalization(
    tokenizer,
    examples,
    padding=True,
    truncation=True,
    max_length=32,
)

for text, input_ids, capitalization_ids in zip(
    examples,
    batch["input_ids"],
    batch["capitalization_ids"],
):
    print("\n" + text)
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    for token, cap_id in zip(tokens, capitalization_ids):
        if token != tokenizer.pad_token:
            print(f"{token:>12}  cap_id={cap_id}")